This is what ChatGPT generated for me, so I will leave it here, but it will not work now.

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from tabpfn import TabPFNClassifier

# Expression CSVs: patient IDs in the first column, probe IDs as other columns.
x_train_df = pd.read_csv("GSE2034_expression.csv", index_col=0)
x_test_df  = pd.read_csv("GSE7390_expression.csv", index_col=0)

y_train_df = pd.read_csv("GSE2034_labels.csv", index_col=0)
y_test_df  = pd.read_csv("GSE7390_labels.csv", index_col=0)

# Keep only probes present in both cohorts, in the same order.
shared_probes = x_train_df.columns.intersection(x_test_df.columns)
x_train_df = x_train_df[shared_probes]
x_test_df = x_test_df[shared_probes]

# Keep only patients whose endpoint was resolved and label is 0 or 1.
train_ok = y_train_df["metastasis_label"].isin([0, 1])
test_ok  = y_test_df["metastasis_label"].isin([0, 1])
x_train_df = x_train_df.loc[train_ok]
y_train = y_train_df.loc[train_ok, "metastasis_label"].astype(int)
x_test_df = x_test_df.loc[test_ok]
y_test = y_test_df.loc[test_ok, "metastasis_label"].astype(int)

# TabPFN works best with a limited number of features.
# Fit feature selection on training data only.
selector = SelectKBest(f_classif, k=min(500, x_train_df.shape[1]))
X_train = selector.fit_transform(x_train_df, y_train)
X_test = selector.transform(x_test_df)

clf = TabPFNClassifier()
clf.fit(X_train, y_train)
print("Test accuracy:", clf.score(X_test, y_test))

This downloads the data from GSE2034 as a csv.

I created this folder to download the csv /content/GSE2034

In [1]:
import csv
import gzip
import re
import urllib.request
from pathlib import Path

import pandas as pd


out_dir = Path(r"/content/GSE2034")
out_dir.mkdir(exist_ok=True)

# Download the processed expression matrix.
matrix_url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE2nnn/GSE2034/"
    "matrix/GSE2034_series_matrix.txt.gz"
)
matrix_path = out_dir / "GSE2034_series_matrix.txt.gz"
urllib.request.urlretrieve(matrix_url, matrix_path)

with gzip.open(matrix_path, "rt", encoding="utf-8") as f:
    expression = pd.read_csv(f, sep="\t", comment="!", index_col=0)

# Download the family SOFT file, which includes the patient clinical table.
soft_url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE2nnn/GSE2034/"
    "soft/GSE2034_family.soft.gz"
)
soft_path = out_dir / "GSE2034_family.soft.gz"
urllib.request.urlretrieve(soft_url, soft_path)


def normalize(text):
    """Normalize a column name for matching despite spelling/punctuation differences."""
    return re.sub(r"[^a-z0-9]+", "", text.lower())


# Find the clinical table header and read its tab-separated rows.
clinical_rows = []
clinical_columns = None

with gzip.open(soft_path, "rt", encoding="utf-8", errors="replace") as f:
    reader = csv.reader(f, delimiter="\t")

    for row in reader:
        normalized = [normalize(cell) for cell in row]

        has_pid = "pid" in normalized
        has_geo_accession = any(
            cell.startswith("geo") and "cession" in cell
            for cell in normalized
        )
        has_relapse = any("relapse" in cell for cell in normalized)

        if has_pid and has_geo_accession and has_relapse:
            clinical_columns = [cell.strip() for cell in row]
            break

    if clinical_columns is None:
        raise RuntimeError("Could not find the clinical table header in the SOFT file.")

    for row in reader:
        # Stop when the SOFT table ends or another record begins.
        if not row or row[0].startswith(("!", "^")):
            break

        if len(row) == len(clinical_columns) and row[0].strip().isdigit():
            clinical_rows.append([cell.strip() for cell in row])

clinical = pd.DataFrame(clinical_rows, columns=clinical_columns)

# Find the GSM and relapse columns, allowing for minor header spelling differences.
normalized_columns = {normalize(col): col for col in clinical.columns}

geo_col = next(
    col for key, col in normalized_columns.items()
    if key.startswith("geo") and "cession" in key
)
relapse_col = next(
    col for key, col in normalized_columns.items()
    if "relapse" in key and ("true" in key or key == "relapse")
)

# Align each clinical row with the corresponding expression-matrix sample.
clinical = clinical.rename(columns={geo_col: "GSM"})
clinical["GSM"] = clinical["GSM"].astype(str)
clinical = clinical.set_index("GSM")

expression.columns = expression.columns.astype(str)
labels_aligned = clinical.reindex(expression.columns)

# Transpose: rows become samples (GSM IDs); columns become probe IDs.
expression_by_sample = expression.T
expression_by_sample.index.name = "sample"

# Optionally include the relapse label as an additional column.
sample_table = expression_by_sample.join(labels_aligned[[relapse_col]])

# Save with the sample ID as the first column.
sample_table.to_csv(
    out_dir / "GSE2034_samples_by_probes.csv",
    index=True,
    index_label="sample",
)

if labels_aligned["GSM" if "GSM" in labels_aligned.columns else relapse_col].isna().any():
    print("Warning: some expression samples did not match a clinical-table row.")

# Save both the full clinical table and the relapse labels aligned to matrix columns.
# clinical.to_csv(out_dir / "GSE2034_clinical.csv")
# labels_aligned.to_csv(out_dir / "GSE2034_clinical_aligned_to_expression.csv")

# The binary target, in the same sample order as expression.columns.
relapse_labels = pd.to_numeric(labels_aligned[relapse_col], errors="coerce")

print("Expression matrix shape:", expression.shape)
print("Clinical table shape:", clinical.shape)
print("Relapse label counts:")
print(relapse_labels.value_counts(dropna=False))


# Download GSE7390 expression data and metastasis labels.
out_dir = Path(r"/content/GSE7390")
out_dir.mkdir(exist_ok=True)

gse7390_matrix_url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE7nnn/GSE7390/"
    "matrix/GSE7390_series_matrix.txt.gz"
)
gse7390_matrix_path = out_dir / "GSE7390_series_matrix.txt.gz"
urllib.request.urlretrieve(gse7390_matrix_url, gse7390_matrix_path)

with gzip.open(gse7390_matrix_path, "rt", encoding="utf-8") as f:
    expression_7390 = pd.read_csv(
        f, sep="\t", comment="!", index_col=0
    )

# Download the SOFT file, which contains sample-level clinical characteristics.
gse7390_soft_url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE7nnn/GSE7390/"
    "soft/GSE7390_family.soft.gz"
)
gse7390_soft_path = out_dir / "GSE7390_family.soft.gz"
urllib.request.urlretrieve(gse7390_soft_url, gse7390_soft_path)

# Collect e.tdm (distant-metastasis event indicator) for each GSM sample.
metastasis_by_sample = {}
current_sample = None

with gzip.open(
    gse7390_soft_path, "rt", encoding="utf-8", errors="replace"
) as f:
    for line in f:
        line = line.strip()

        if line.startswith("^SAMPLE ="):
            current_sample = line.split("=", 1)[1].strip()

        elif line.startswith("^"):
            current_sample = None

        elif current_sample and line.startswith("!Sample_characteristics_"):
            characteristic = line.split("=", 1)[1].strip()
            key, separator, value = characteristic.partition(":")

            if separator and key.strip().lower() == "e.tdm":
                metastasis_by_sample[current_sample] = value.strip()

# Align labels to the expression matrix's sample columns.
expression_7390.columns = expression_7390.columns.astype(str)
metastasis_labels = pd.Series(
    metastasis_by_sample, name="e.tdm"
).reindex(expression_7390.columns)

metastasis_labels = pd.to_numeric(metastasis_labels, errors="coerce")

if metastasis_labels.isna().any():
    missing_samples = metastasis_labels[metastasis_labels.isna()].index.tolist()
    raise RuntimeError(
        f"Could not find e.tdm labels for these samples: {missing_samples[:10]}"
    )

# Transpose so each row is a sample and each probe is a column.
samples_7390 = expression_7390.T
samples_7390.index.name = "sample"
samples_7390["e.tdm"] = metastasis_labels

# Save sample ID first, followed by probe columns and the metastasis label.
samples_7390.to_csv(
    out_dir / "GSE7390_samples_by_probes.csv",
    index=True,
    index_label="sample",
)

print("GSE7390 expression matrix shape:", expression_7390.shape)
print("GSE7390 sample table shape:", samples_7390.shape)
print("GSE7390 e.tdm counts:")
print(samples_7390["e.tdm"].value_counts(dropna=False))

Expression matrix shape: (22283, 286)
Clinical table shape: (286, 6)
Relapse label counts:
relapse (1=True)
0    179
1    107
Name: count, dtype: int64
GSE7390 expression matrix shape: (22283, 198)
GSE7390 sample table shape: (198, 22284)
GSE7390 e.tdm counts:
e.tdm
0    147
1     51
Name: count, dtype: int64


Test run for using GSE2034 data for train and test

In [ ]:
## Base library Installation
# Install Baselines for model comparison
!uv pip install catboost xgboost

# Install the datasets library for loading example data
!uv pip install datasets

# Install rich for better and more readable printing
!uv pip install rich

## TabPFN Installation optimized for Google Colab
# Install the TabPFN Client library
!uv pip install tabpfn-client

# Install TabPFN extensions for additional functionalities
!uv pip install 'tabpfn-extensions[all]'

# Install tabpfn
!uv pip install tabpfn

# Install finetunedtabpfn
!uv pip install tabpfn tabpfn-extensions --upgrade

# This is needed for finetunedtabpfn not to have errors while running
!uv pip uninstall -y pyopenssl cryptography tabpfn tabpfn-extensions
!uv pip install pyopenssl cryptography tabpfn

Using Python 3.13.16 environment at: /usr
Checked 2 packages in 106ms
Using Python 3.13.16 environment at: /usr
Resolved 35 packages in 158ms
Uninstalled 2 packages in 3ms
Installed 2 packages in 6ms
 - fsspec==2026.9.0
 + fsspec==2026.2.0
 - huggingface-hub==2.4.0
 + huggingface-hub==1.33.0
Using Python 3.13.16 environment at: /usr
Checked 1 package in 95ms
Using Python 3.13.16 environment at: /usr
Resolved 48 packages in 243ms
Uninstalled 5 packages in 56ms
Installed 5 packages in 40ms
 - pandas==3.1.1
 + pandas==2.3.3
 - pydantic==2.14.2
 + pydantic==2.14.1
 - pydantic-core==2.50.2
 + pydantic-core==2.50.1
 - pydantic-settings==2.16.2
 + pydantic-settings==2.15.0
 - scikit-learn==1.9.1
 + scikit-learn==1.9.0
Using Python 3.13.16 environment at: /usr
Checked 1 package in 103ms
Using Python 3.13.16 environment at: /usr
Checked 1 package in 99ms
Using Python 3.13.16 environment at: /usr
Resolved 81 packages in 249ms
Prepared 7 packages in 0.89ms
Uninstalled 7 packages in 43ms
Installed

In [ ]:
# Authentication code - No need to read it!
import os
import getpass
from rich.console import Console
from rich.rule import Rule
from rich.panel import Panel

console = Console()
# --- Prior Labs Authentication ---
console.print(Rule("[bold]Prior Labs Authentication[/bold]"))
console.print(
    "\nTabPFN model weights or API access via client requires a free "
    "[bold]Prior Labs account[/bold] and acceptance of the non-commercial "
    "license.\n"
)

tabpfn_token = None

# 1. Try Colab secret TABPFN_TOKEN
try:
    from google.colab import userdata
    tabpfn_token = userdata.get("TABPFN_TOKEN")
    if tabpfn_token:
        os.environ["TABPFN_TOKEN"] = tabpfn_token
        console.print("[bold green]✅ Found TABPFN_TOKEN in Colab secrets.[/bold green]")
except Exception:
    pass

# 2. If no token found, prompt the user
if not tabpfn_token:
    console.print(
        Panel(
            "To get your access token:\n\n"
            "  1. Go to [link=https://ux.priorlabs.ai]ux.priorlabs.ai[/link] and sign up / log in\n"
            "  2. Accept the license at [link=https://ux.priorlabs.ai/account/licenses]ux.priorlabs.ai/account/licenses[/link]\n"
            "  3. Copy your Access Token from [link=https://ux.priorlabs.ai/account]ux.priorlabs.ai/account[/link]\n\n"
            "[bold yellow]Tip:[/bold yellow] Save the token as a Colab secret named "
            "[bold cyan]TABPFN_TOKEN[/bold cyan] to skip this step next time.",
            title="[bold]🔑 Prior Labs Access Token required",
            border_style="blue",
        )
    )
    while not tabpfn_token:
        token_input = getpass.getpass("Paste your TABPFN_TOKEN and press Enter: ")
        if token_input.strip():
            tabpfn_token = token_input.strip()
            os.environ["TABPFN_TOKEN"] = tabpfn_token
        else:
            console.print("[red]Token cannot be empty. Please try again.[/red]")



console.print(Panel.fit("[bold magenta]TabPFN Demo: Backend Selection[/bold magenta]"))
console.print("\nThis script can run TabPFN using one of two backends:")
console.print("  [bold]local:[/bold] Uses a local GPU (NVIDIA). Requires CUDA.")
console.print(
    "  [bold]client:[/bold] Uses the TabPFN API. Requires an internet connection and a free account."
)

backend = None
while backend is None:
  console.print(
      "\n[bold]Choose your backend[/bold]: - If no text box is shown, restart the cell.",
  )
  user_input = input("Enter 'local' or 'client' and press return:")
  if user_input not in ["local", "client"]:
    continue
  backend = user_input

console.print(
    f"\n✅ You have selected the '[bold green]{backend}[/bold green]' backend."
)

console.print(Rule(f"[bold]Setting up [cyan]{backend}[/cyan] backend[/bold]"))

if backend == "local":
    console.print("Attempting local backend setup...")
    import torch

    if not torch.cuda.is_available():
        console.print(
            "[bold red]Error:[/bold red] GPU device not found. For fast training, please enable GPU.",
            style="red",
        )
        console.print(
            "In Colab: Go to [bold]Runtime -> Change runtime type -> Hardware accelerator -> GPU.[/bold]",
            style="yellow",
        )
        raise SystemError("GPU device not found.")
    console.print("[bold green]✅ GPU is available.[/bold green]")




    console.print("")
    console.print("Importing local TabPFN library...")

    from tabpfn import TabPFNClassifier, TabPFNRegressor

    console.print("[bold green]✅ TabPFN (local) imported successfully.[/bold green]")
elif backend == "client":
    console.print("Attempting client backend setup...")
    console.print("Importing TabPFN client library...")
    from tabpfn_client import TabPFNClassifier, TabPFNRegressor, init

    init()
    console.print("[bold green]✅ TabPFN (client) initialized.[/bold green]")


──────────────────────────────────────────── Prior Labs Authentication ────────────────────────────────────────────

TabPFN model weights or API access via client requires a free Prior Labs account and acceptance of the 
non-commercial license.

✅ Found TABPFN_TOKEN in Colab secrets.

╭────────────────────────────────╮
│ TabPFN Demo: Backend Selection │
╰────────────────────────────────╯

This script can run TabPFN using one of two backends:

local: Uses a local GPU (NVIDIA). Requires CUDA.

... (5 outputs left out)

Attempting local backend setup...

✅ GPU is available.

Importing local TabPFN library...

✅ TabPFN (local) imported successfully.

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from tabpfn import TabPFNClassifier

df = pd.read_csv("/content/GSE2034/GSE2034_samples_by_probes.csv", index_col=0)

# Expression CSVs: patient IDs in the first column, probe IDs as other columns.
x = df.drop(["relapse (1=True)"], axis=1).astype(float)
# x_test_df  = pd.read_csv(r".Data\GSE7390\GSE7390_expression.csv", index_col=0)

y = df["relapse (1=True)"].astype(int)
# y_test_df  = pd.read_csv(r".Data\GSE7390\GSE7390_labels.csv", index_col=0)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20, random_state=42
)

print("Shape of data:\n")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}\n")

# Train and evaluate the TabPFN classifier
tabpfn_classifier = TabPFNClassifier(random_state=42, ignore_pretraining_limits=True)
print("Training TabPFN classifier...")
tabpfn_classifier.fit(X_train, y_train)
predictions = tabpfn_classifier.predict(X_test)
print(f"predictions (shape {predictions.shape}): {predictions}")
probas = tabpfn_classifier.predict_proba(X_test)
print(f"probabilities (shape {probas.shape}): {probas}")
accuracy = tabpfn_classifier.score(X_test, y_test)
print(f"Test accuracy: {accuracy}")

Shape of data:

X_train: (228, 22283)
X_test: (58, 22283)
y_train: (228,)
y_test: (58,)

Training TabPFN classifier...


/usr/local/lib/python3.13/dist-packages/tabpfn/classifier.py:870: UserWarning: Running 8 estimators covers at most 6144 of 22283 features (max_features_per_estimator=768); the remaining features are never sampled by any ensemble member. Pass n_estimators >= 30 to cover all features.
  self.n_estimators_ = scale_n_estimators_for_feature_coverage(


predictions (shape (58,)): [0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 1 0 1 0 1 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 1]
probabilities (shape (58, 2)): [[0.62613964 0.37386036]
 [0.77436465 0.22563536]
 [0.53285503 0.46714497]
 [0.39631805 0.6036819 ]
 [0.7495529  0.25044712]
 [0.7572274  0.24277261]
 [0.7094536  0.29054645]
 [0.55699074 0.44300926]
 [0.652549   0.34745097]
 [0.7701876  0.22981238]
 [0.71222436 0.28777564]
 [0.52365315 0.47634685]
 [0.49017325 0.5098268 ]
 [0.812338   0.18766199]
 [0.6914918  0.30850825]
 [0.30487442 0.6951256 ]
 [0.7006106  0.29938942]
 [0.6801665  0.31983358]
 [0.78633237 0.21366763]
 [0.72799945 0.2720006 ]
 [0.44689983 0.5531002 ]
 [0.7946002  0.20539981]
 [0.4525419  0.5474582 ]
 [0.63027906 0.36972094]
 [0.46089888 0.5391011 ]
 [0.58867586 0.41132414]
 [0.69329226 0.30670774]
 [0.6654624  0.33453766]
 [0.50283235 0.49716765]
 [0.5025578  0.4974422 ]
 [0.80879366 0.19120634]
 [0.58326745 0.41673258]
 [0.6970938  0.30290624

Fine tunned variant

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from tabpfn.finetuning import FinetunedTabPFNClassifier

# Initialize the fine-tuning classifier
tabpfn_classifier = FinetunedTabPFNClassifier(
    epochs=30,               # Default is 30 (range: 10 to 100)
    learning_rate=1e-5,                 # Learning rate for weight updates
    device='cuda',           # Strongly recommended to use a GPU ('cuda' or 'mps')
    random_state=42
)


df = pd.read_csv("/content/GSE2034/GSE2034_samples_by_probes.csv", index_col=0)

# Expression CSVs: patient IDs in the first column, probe IDs as other columns.
x = df.drop(["relapse (1=True)"], axis=1).astype(float)
# x_test_df  = pd.read_csv(r".Data\GSE7390\GSE7390_expression.csv", index_col=0)

y = df["relapse (1=True)"].astype(int)
# y_test_df  = pd.read_csv(r".Data\GSE7390\GSE7390_labels.csv", index_col=0)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20, random_state=42
)

print("Shape of data:\n")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}\n")

# Train and evaluate the TabPFN classifier
print("Training TabPFN classifier...")
tabpfn_classifier.fit(X_train, y_train)
predictions = tabpfn_classifier.predict(X_test)
print(f"predictions (shape {predictions.shape}): {predictions}")
probas = tabpfn_classifier.predict_proba(X_test)
print(f"probabilities (shape {probas.shape}): {probas}")
accuracy = tabpfn_classifier.score(X_test, y_test)
print(f"Test accuracy: {accuracy}")

/usr/local/lib/python3.13/dist-packages/tabpfn/finetuning/finetuned_classifier.py:175: UserWarning: `use_fixed_preprocessing_seed` should only be used if `n_estimators_finetune` == `n_estimators_validation` == `n_estimators_final_inference`. Consider setting the number of estimators for validation and final inference to the same value as `n_estimators_finetune`(=2).
  super().__init__(


Shape of data:

X_train: (228, 22283)
X_test: (58, 22283)
y_train: (228,)
y_test: (58,)

Training TabPFN classifier...


/usr/local/lib/python3.13/dist-packages/tabpfn/finetuning/finetuned_classifier.py:424: UserWarning: `output_dir` is not set. This means no checkpointing will be done and all progress will be lost if the training is interrupted.
  super().fit(X, y, X_val=X_val, y_val=y_val, output_dir=output_dir)


TypeError: Config() got an unexpected keyword argument 'deprecated'